# ❄️ SNOW Data Product Pipeline
## Domain: IT Service Management | Catalog: `snow_product` | Owner: ITSM Team

This notebook demonstrates the **SNOW Data Product** — owned and managed by the IT Service Management team within their own catalog (`snow_product`).

**Data Product Characteristics shown here:**
- ✅ Clear Owner — ITSM Team owns the catalog
- ✅ Business Purpose — Service health metrics for operational reviews and SLA tracking
- ✅ Documented Definitions — Every column has a COMMENT
- ✅ Quality Checks — Priority validation, SLA compliance ranges, resolution metrics
- ✅ Lineage — Bronze → Silver → Gold transformation chain
- ✅ Reliability — SLA: 15-minute refresh for incidents

**Data Objects:**
- `raw_incidents` — 12 IT incidents from ServiceNow
- `raw_change_requests` — 8 change requests from ServiceNow

**Cross-Domain Link:** The `affected_application_id` field in SNOW data maps to `app_id` in `adoit_product` — enabling federated consumption.

## 🪨 Bronze Layer — Raw Ingestion from ServiceNow
Raw ticket data extracted via ServiceNow Table API. Preserves original field values.

In [0]:
# ── Bootstrap: Create bronze tables and load CSV data ──
import os, pandas as pd

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
bundle_root = notebook_path.rsplit('/src/notebooks/', 1)[0]
data_dir = f"/Workspace{bundle_root}/src/data"

if not os.path.exists(data_dir):
    data_dir = "/Workspace/Users/skarotirajashekar@godevsuite060.onmicrosoft.com/data-mesh-demo/src/data"

print(f"Data directory: {data_dir}")

for csv_file, table_name in [
    ("snow_incidents.csv", "snow_product.bronze.raw_incidents"),
    ("snow_change_requests.csv", "snow_product.bronze.raw_change_requests"),
]:
    pdf = pd.read_csv(f"{data_dir}/{csv_file}")
    df = spark.createDataFrame(pdf)
    df.write.mode("overwrite").saveAsTable(table_name)
    print(f"  ✓ {table_name} ({len(pdf)} rows)")

print("Bronze tables loaded.")

In [0]:
%sql
-- Create silver and gold tables (empty) so MERGE INTO has targets
-- Silver: incidents (columns match MERGE source)
CREATE TABLE IF NOT EXISTS snow_product.silver.incidents (
  incident_id STRING, short_description STRING, priority INT, priority_label STRING,
  state STRING, category STRING, assigned_to STRING, assignment_group STRING,
  affected_application_id STRING, created_at TIMESTAMP, resolved_at TIMESTAMP,
  resolution_hours DOUBLE, sla_breached BOOLEAN, is_open BOOLEAN,
  processed_at TIMESTAMP
);

-- Silver: change_requests (columns match MERGE source)
CREATE TABLE IF NOT EXISTS snow_product.silver.change_requests (
  change_id STRING, description STRING, change_type STRING, risk STRING,
  state STRING, affected_application_id STRING, requested_by STRING,
  planned_start TIMESTAMP, planned_end TIMESTAMP, success BOOLEAN,
  duration_hours DOUBLE, processed_at TIMESTAMP
);

-- Gold: service_health (columns match MERGE source exactly)
CREATE TABLE IF NOT EXISTS snow_product.gold.service_health (
  affected_application_id STRING, total_incidents INT, open_incidents INT,
  critical_incidents INT, high_incidents INT, sla_breaches INT,
  sla_compliance_pct DOUBLE, avg_resolution_hours DOUBLE,
  total_changes INT, successful_changes INT, failed_changes INT,
  change_success_rate_pct DOUBLE, emergency_changes INT,
  risk_score STRING, product_generated_at TIMESTAMP
);

In [0]:
%sql
-- Bronze: Raw incidents from ServiceNow
SELECT incident_id, short_description, priority, state, category, 
       assigned_to, affected_application_id, created_at, resolved_at, sla_breached
FROM snow_product.bronze.raw_incidents
ORDER BY created_at DESC;

In [0]:
%sql
-- Bronze: Raw change requests from ServiceNow
SELECT change_id, description, change_type, risk, state,
       affected_application_id, requested_by, success
FROM snow_product.bronze.raw_change_requests
ORDER BY planned_start DESC;

## ⚙️ Silver Layer — Curated & Enriched
Transformations applied:
1. **Derived Fields** — `priority_label` (human-readable), `resolution_hours`, `is_open`
2. **Standardization** — `TRIM()`, `INITCAP()` on text fields
3. **Calculated Metrics** — `planned_duration_hours`, `actual_duration_hours`, `on_schedule`
4. **Validation** — Priority 1–4, non-null incident IDs

In [0]:
%sql
-- Silver transformation for incidents
MERGE INTO snow_product.silver.incidents AS target
USING (
    SELECT 
        incident_id, TRIM(short_description) AS short_description, priority,
        CASE priority WHEN 1 THEN 'Critical' WHEN 2 THEN 'High' WHEN 3 THEN 'Medium' WHEN 4 THEN 'Low' END AS priority_label,
        state, INITCAP(category) AS category, assigned_to, assignment_group, affected_application_id,
        created_at, resolved_at,
        ROUND(TIMESTAMPDIFF(MINUTE, created_at, resolved_at) / 60.0, 2) AS resolution_hours,
        sla_breached,
        CASE WHEN state IN ('New', 'In Progress', 'On Hold') THEN TRUE ELSE FALSE END AS is_open,
        current_timestamp() AS processed_at
    FROM snow_product.bronze.raw_incidents
    WHERE incident_id IS NOT NULL
) AS source
ON target.incident_id = source.incident_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
%sql
-- Silver transformation for change requests
MERGE INTO snow_product.silver.change_requests AS target
USING (
    SELECT 
        change_id, TRIM(description) AS description, change_type, risk,
        state, affected_application_id, requested_by,
        planned_start, planned_end, success,
        ROUND(TIMESTAMPDIFF(MINUTE, planned_start, planned_end) / 60.0, 2) AS duration_hours,
        current_timestamp() AS processed_at
    FROM snow_product.bronze.raw_change_requests
    WHERE change_id IS NOT NULL
) AS source
ON target.change_id = source.change_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
%sql
-- Silver incidents with derived metrics
SELECT incident_id, short_description, priority_label, state, category,
       affected_application_id, resolution_hours, sla_breached, is_open
FROM snow_product.silver.incidents
ORDER BY priority, created_at DESC;

## 🏆 Gold Layer — Service Health Data Product
Aggregates incidents and changes **per application** to produce the **Service Health** data product. This is what IT Operations and management consume for SLA reviews.

In [0]:
%sql
-- Gold: Build Service Health data product by aggregating per application
MERGE INTO snow_product.gold.service_health AS target
USING (
    WITH incident_metrics AS (
        SELECT 
            affected_application_id,
            COUNT(*) AS total_incidents,
            SUM(CASE WHEN is_open THEN 1 ELSE 0 END) AS open_incidents,
            SUM(CASE WHEN priority = 1 THEN 1 ELSE 0 END) AS critical_incidents,
            SUM(CASE WHEN priority = 2 THEN 1 ELSE 0 END) AS high_incidents,
            SUM(CASE WHEN sla_breached THEN 1 ELSE 0 END) AS sla_breaches,
            ROUND(100.0 * SUM(CASE WHEN NOT sla_breached THEN 1 ELSE 0 END) / COUNT(*), 1) AS sla_compliance_pct,
            ROUND(AVG(resolution_hours), 2) AS avg_resolution_hours
        FROM snow_product.silver.incidents
        WHERE affected_application_id IS NOT NULL
        GROUP BY affected_application_id
    ),
    change_metrics AS (
        SELECT 
            affected_application_id,
            COUNT(*) AS total_changes,
            SUM(CASE WHEN success = TRUE THEN 1 ELSE 0 END) AS successful_changes,
            SUM(CASE WHEN success = FALSE THEN 1 ELSE 0 END) AS failed_changes,
            ROUND(100.0 * SUM(CASE WHEN success = TRUE THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 1) AS change_success_rate_pct,
            SUM(CASE WHEN change_type = 'Emergency' THEN 1 ELSE 0 END) AS emergency_changes
        FROM snow_product.silver.change_requests
        WHERE affected_application_id IS NOT NULL
        GROUP BY affected_application_id
    )
    SELECT 
        COALESCE(i.affected_application_id, c.affected_application_id) AS affected_application_id,
        COALESCE(i.total_incidents, 0) AS total_incidents,
        COALESCE(i.open_incidents, 0) AS open_incidents,
        COALESCE(i.critical_incidents, 0) AS critical_incidents,
        COALESCE(i.high_incidents, 0) AS high_incidents,
        COALESCE(i.sla_breaches, 0) AS sla_breaches,
        COALESCE(i.sla_compliance_pct, 100.0) AS sla_compliance_pct,
        i.avg_resolution_hours,
        COALESCE(c.total_changes, 0) AS total_changes,
        COALESCE(c.successful_changes, 0) AS successful_changes,
        COALESCE(c.failed_changes, 0) AS failed_changes,
        COALESCE(c.change_success_rate_pct, 100.0) AS change_success_rate_pct,
        COALESCE(c.emergency_changes, 0) AS emergency_changes,
        CASE 
            WHEN COALESCE(i.critical_incidents, 0) >= 2 OR COALESCE(i.sla_breaches, 0) >= 2 THEN 'Critical'
            WHEN COALESCE(i.critical_incidents, 0) >= 1 OR COALESCE(i.high_incidents, 0) >= 2 THEN 'High'
            WHEN COALESCE(i.high_incidents, 0) >= 1 THEN 'Medium'
            ELSE 'Low'
        END AS risk_score,
        current_timestamp() AS product_generated_at
    FROM incident_metrics i
    FULL OUTER JOIN change_metrics c ON i.affected_application_id = c.affected_application_id
) AS source
ON target.affected_application_id = source.affected_application_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
%sql
-- The final data product: Service Health per application
SELECT affected_application_id, total_incidents, open_incidents, critical_incidents,
       sla_breaches, sla_compliance_pct, ROUND(avg_resolution_hours, 1) AS avg_hours,
       total_changes, change_success_rate_pct, risk_score
FROM snow_product.gold.service_health
ORDER BY risk_score, total_incidents DESC;

In [0]:
%sql
-- Data quality validation rules for SNOW product
SELECT 
    'Completeness: assigned_to' AS check_name,
    ROUND(COUNT(CASE WHEN assigned_to IS NOT NULL THEN 1 END) * 100.0 / COUNT(*), 1) AS score_pct,
    CASE WHEN COUNT(CASE WHEN assigned_to IS NOT NULL THEN 1 END) * 100.0 / COUNT(*) >= 90 THEN 'PASS' ELSE 'FAIL' END AS status
FROM snow_product.silver.incidents
UNION ALL
SELECT 'Validity: priority range (1-4)',
    ROUND(COUNT(CASE WHEN priority BETWEEN 1 AND 4 THEN 1 END) * 100.0 / COUNT(*), 1),
    CASE WHEN COUNT(CASE WHEN priority NOT BETWEEN 1 AND 4 THEN 1 END) = 0 THEN 'PASS' ELSE 'FAIL' END
FROM snow_product.silver.incidents
UNION ALL
SELECT 'Validity: SLA compliance 0-100%',
    CASE WHEN COUNT(CASE WHEN sla_compliance_pct BETWEEN 0 AND 100 THEN 1 END) = COUNT(*) THEN 100.0 ELSE 0 END,
    CASE WHEN COUNT(CASE WHEN sla_compliance_pct < 0 OR sla_compliance_pct > 100 THEN 1 END) = 0 THEN 'PASS' ELSE 'FAIL' END
FROM snow_product.gold.service_health
UNION ALL
SELECT 'Cross-Domain: app_id coverage',
    ROUND(COUNT(CASE WHEN affected_application_id IS NOT NULL THEN 1 END) * 100.0 / COUNT(*), 1),
    'INFO'
FROM snow_product.silver.incidents;

In [0]:
%sql
-- Data product metadata embedded in Unity Catalog
SHOW TBLPROPERTIES snow_product.gold.service_health;